Visualisation for research question 2

All the relevant imports:

In [21]:
import pandas as pd
import plotly.graph_objects as go






In [22]:
# Reading in our CSV File with the Data
df = pd.read_csv("rq_2_berlin_houspa_pollution_2023_2025_diff.csv")

Convert column "date" to datetime:

In [23]:
df["date"] = pd.to_datetime(df["date"])

We create a scatterplot which shows how the increase or decrease of the pollution 
effects the amount of house sparrows in berlin hotspots

In [25]:

# Removing the NaN values from the first day
df = df.dropna(subset=["diff_howMany", "diff_pm2_5_mean", "diff_no2_mean"])

# Define strong changes in pollution as the top 5 % of absolut pollution changes
pm25_threshold = df["diff_pm2_5_mean"].abs().quantile(0.95)
no2_threshold = df["diff_no2_mean"].abs().quantile(0.95)

# Just keep the rows where the pollution change is defined as strong
df_filtered = df[
    (df["diff_pm2_5_mean"].abs() >= pm25_threshold) |
    (df["diff_no2_mean"].abs() >= no2_threshold)
]

# Calculate symmetric axis limits so the plot looks balanced around zero
birds_max = df_filtered["diff_howMany"].abs().max()
pollution_max = max(
    df_filtered["diff_pm2_5_mean"].abs().max(),
    df_filtered["diff_no2_mean"].abs().max()
)

# Extra space from the boarder 
birds_limit = birds_max * 1.1
pollution_limit = pollution_max * 1.1

# Create empty Figure
fig = go.Figure()

# Adding Hose Sparrow Changes on left y-axis  
fig.add_trace(go.Scatter(
    x=df_filtered["date"],
    y=df_filtered["diff_howMany"],
    mode="markers",
    name="Δ House Sparrows",
    marker=dict(
        size=6,
        color="#5DA5DA"
    ),
    # § LLM Help: for creating Hover Label
    hovertemplate=(
        "<b>Δ House Sparrows</b><br>"
        "Date: %{x|%Y-%m-%d}<br>"
        "Change: %{y}<extra></extra>"
    )
))

# Adding pm2.5 pollution Changes on right y-axis 
fig.add_trace(go.Scatter(
    x=df_filtered["date"],
    y=df_filtered["diff_pm2_5_mean"],
    mode="markers",
    name="Δ PM2.5",
    yaxis="y2",
    marker=dict(
        size=6,
        color="#ff6fb5",
        opacity=0.8
    ),
     # § LLM Help: for creating Hover Label
    hovertemplate=(
        "<b>Δ PM2.5</b><br>"
        "Date: %{x|%Y-%m-%d}<br>"
        "Change: %{y:.2f}<extra></extra>"
    )
))

# Adding no2 pollution Changes on right y-axis 
fig.add_trace(go.Scatter(
    x=df_filtered["date"],
    y=df_filtered["diff_no2_mean"],
    mode="markers",
    name="Δ NO2",
    yaxis="y2",
    marker=dict(
        size=6,
        color="#F6E27A",
        opacity=0.8
    ),
    # § LLM Help: for creating Hover Label
    hovertemplate=(
        "<b>Δ NO2</b><br>"
        "Date: %{x|%Y-%m-%d}<br>"
        "Change: %{y:.2f}<extra></extra>"
    )
))

# Create the Layout of the figure
fig.update_layout(
    
    # titel
    title=dict(
        text="Days with Strong Pollution Changes: House Sparrows and Air Pollution in Berlin (2023–2025)",
        x=0.5
    ),
    
    # x-axis (Date)
    xaxis=dict(
        title="Date",
        showgrid=True,
        gridcolor="rgba(0,0,0,0.08)"
    ),

    # Left y-axis (House Sparrow)
    yaxis=dict(
        title="Change in House Sparrow Count",
        showgrid=True,
        gridcolor="rgba(0,0,0,0.08)",
        zeroline=True,
        zerolinecolor="rgba(0,0,0,0.25)",
        range=[-birds_limit, birds_limit],
        nticks=7
    ),

    # Right y-axis (Pollution)
    yaxis2=dict(
        title="Change in Pollution (PM2.5 / NO2)",
        overlaying="y",
        side="right",
        range=[pollution_limit, -pollution_limit],
        nticks=7,
        showgrid=False,
        zeroline=False
    ),
    
    # Setting general layout and styling for the figure
    template="plotly_white",
    width=1100,
    height=600,
    font=dict(size=14),

    # Creating the legend
    legend=dict(
        title="Variable",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),

    # Creating an interactive dropdown menu to change between 
    # "Show all", "Birds + PM2_5", "Birds only" and "Birds + NO2"
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=1.02,
            y=1.18,
            showactive=True,
            buttons=[
                dict(
                    label="Show all",
                    method="update",
                    args=[
                        {"visible": [True, True, True]},
                        {"title":{"text": "Days with Strong Pollution Changes: House Sparrows and Air Pollution in Berlin (2023–2025)","x": 0.5}}
                    ]
                ),
                dict(
                    label="Birds only",
                    method="update",
                    args=[
                        {"visible": [True, False, False]},
                        {"title":{"text": "Days with Strong Pollution Changes: House Sparrow Count","x": 0.5}}
                    ]
                ),
                dict(
                    label="PM2.5 + Birds",
                    method="update",
                    args=[
                        {"visible": [True, True, False]},
                        {"title":{"text": "Days with Strong PM2.5 Changes and House Sparrow Count","x": 0.5}}
                    ]
                ),
                dict(
                    label="NO2 + Birds",
                    method="update",
                    args=[
                        {"visible": [True, False, True]},
                        {"title":{"text": "Days with Strong NO2 Changes and House Sparrow Count","x": 0.5}}
                    ]
                    
                )
                
            ]
        )
    ],
    
    # Setting margins around the plot
    margin=dict(l=70, r=140, t=160, b=70)
)

fig.show()